In [ ]:
## Notebook Structure

#1. Library installation and imports
#2. Building data cleaning and risk score construction
#3. Road network extraction from OpenStreetMap
#4. Road-type vulnerability assignment
#5. Baseline network centrality calculation
#6. Integration of neighborhood-level building risk with road segments
#7. T0 immediate post-earthquake closure scenario
#8. T1 partial recovery and reopening logic
#9. Network degradation and stress analysis
#10. Identification of stress corridors, local bottlenecks, and critical open backbone roads
#11. Interactive map generation
#12. GitHub Pages export

In [ ]:
#1. Library installation and imports

!pip install osmnx networkx geopandas folium matplotlib pandas numpy pymupdf requests shapely

In [ ]:
import os
import time
import requests
import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
import folium
import matplotlib.pyplot as plt

from shapely.geometry import shape
from io import StringIO
from IPython.display import Markdown, display

print("Kütüphaneler hazır.")

In [ ]:
display(Markdown("""
# Avcılar Mw 7.5 Ağır Deprem Senaryosu Tabanlı Yol Ağı Stres Analizi

Bu çalışmada Avcılar'daki tüm araç yolları kullanılarak ağır deprem senaryosu altında:

1. Hangi yolların kapanma riski taşıdığı,
2. Kapanmalar sonrası hangi yolların stres altına girdiği,
3. Hangi yollar önce temizlenirse network'ün daha hızlı toparlanacağı

analiz edilecektir.

Bu çalışma gerçek hasar tahmini değildir. İBB Mw 7.5 deprem senaryosu, mahalle bazlı bina verileri ve yol ağı topolojisi kullanılarak oluşturulan bir network stress test çalışmasıdır.
"""))

In [ ]:
#2. Building data cleaning and risk score construction

ibb_avcilar_pdf_url = "https://depremzemin.ibb.istanbul/uploads/prefix-avcilar-666ad4f31841f.pdf"

building_csv_url = "https://data.ibb.gov.tr/dataset/be3582eb-09d7-42f8-84d3-b3817dc9ab0a/resource/cef193d5-0bd2-4e8d-8a69-275c50288875/download/2017-yl-mahalle-bazl-bina-saylar.csv"

display(Markdown("""
## Kullanılacak ana kaynaklar

1. **İBB Avcılar Olası Deprem Kayıp Tahminleri Kitapçığı**
   Mw 7.5 ağır deprem senaryosu için temel dayanak olarak kullanılacak.

2. **Mahalle Bazlı Bina Sayıları Verisi**
   Mahalle bazında bina yaşı ve kat sayısı dağılımını modele eklemek için kullanılacak.
"""))

In [ ]:
pdf_path = "ibb_avcilar_mw75_deprem_kitapcigi.pdf"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(ibb_avcilar_pdf_url, headers=headers)

with open(pdf_path, "wb") as f:
    f.write(response.content)

print("PDF indirildi:", pdf_path)
print("Dosya boyutu:", round(os.path.getsize(pdf_path) / 1024 / 1024, 2), "MB")

In [ ]:
doc = fitz.open(pdf_path)

pdf_pages = []

for page_number, page in enumerate(doc, start=1):
    text = page.get_text()
    pdf_pages.append({
        "page": page_number,
        "text": text
    })

keywords = [
    "Mw",
    "7,5",
    "ağır hasar",
    "çok ağır hasar",
    "altyapı",
    "yol kapanma",
    "ulaşım",
    "mahalle",
    "can kaybı",
    "yaralı",
    "geçici barınma"
]

keyword_hits = []

for item in pdf_pages:
    page = item["page"]
    text_lower = item["text"].lower()

    for keyword in keywords:
        if keyword.lower() in text_lower:
            keyword_hits.append({
                "page": page,
                "keyword": keyword
            })

keyword_results = pd.DataFrame(keyword_hits).drop_duplicates().sort_values(["page", "keyword"])

keyword_results

In [ ]:
# ==============================
# HÜCRE 7 - MAHALLE BAZLI BİNA VERİSİNİ DOĞRU ENCODING İLE ÇEKME
# ==============================

def read_csv_from_url_with_encoding(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print("Veri indirilemedi. Status code:", response.status_code)
        return None

    raw_content = response.content

    encodings = ["utf-8-sig", "utf-8", "cp1254", "iso-8859-9", "latin1"]
    separators = [",", ";"]

    for encoding in encodings:
        for sep in separators:
            try:
                text = raw_content.decode(encoding)
                df = pd.read_csv(StringIO(text), sep=sep)

                if df.shape[1] > 1:
                    print("Başarılı encoding:", encoding)
                    print("Başarılı ayırıcı:", sep)
                    return df

            except Exception:
                pass

    return None

building_df = read_csv_from_url_with_encoding(building_csv_url)

print("Bina verisi yüklendi.")
print("Boyut:", building_df.shape)

display(building_df.head())
display(building_df.columns)

In [ ]:
# ==============================
# HÜCRE 7B - SADECE AVCILAR MAHALLELERİNİ FİLTRELEME
# ==============================

# İlçe adını standartlaştırıyoruz
building_df["ilce_adi_clean"] = (
    building_df["ilce_adi"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Sadece Avcılar ilçesini alıyoruz
avcilar_building_df = building_df[
    building_df["ilce_adi_clean"].str.contains("AVCILAR", na=False)
].copy()

print("Avcılar bina verisi boyutu:", avcilar_building_df.shape)

display(avcilar_building_df)

In [ ]:
# ==============================
# HÜCRE 7C - AVCILAR MAHALLE BAZLI BİNA RİSK SKORU
# ==============================

import re
import unicodedata

def normalize_text(text):
    text = str(text).upper().strip()
    text = text.replace("İ", "I").replace("İ", "I")
    text = text.replace("Ş", "S").replace("Ğ", "G").replace("Ü", "U").replace("Ö", "O").replace("Ç", "C")
    text = text.replace("Â", "A")
    text = re.sub(r"[^A-Z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

building_risk_df = avcilar_building_df.copy()

building_risk_df["mahalle_key"] = building_risk_df["mahalle_adi"].apply(normalize_text)

# Kolon adlarını daha kolay yakalamak için normalize ediyoruz
col_keys = {col: normalize_text(col) for col in building_risk_df.columns}

print("Kolonlar:")
for col, key in col_keys.items():
    print(col, "->", key)

In [ ]:
# ==============================
# HÜCRE 7D - BİNA KAYNAKLI MAHALLE RİSKİNİ SAĞLAM HESAPLAMA
# ==============================

import re
import numpy as np
import pandas as pd

# Orijinal Avcılar bina tablosundan temiz kopya alıyoruz
building_risk_df = avcilar_building_df.copy()

# Metin temizleme fonksiyonu
def normalize_text(text):
    text = str(text).upper().strip()
    text = text.replace("İ", "I").replace("İ", "I")
    text = text.replace("Ş", "S").replace("Ğ", "G").replace("Ü", "U").replace("Ö", "O").replace("Ç", "C")
    text = text.replace("Â", "A")
    text = re.sub(r"[^A-Z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Kolon adlarını temizleme fonksiyonu
def clean_column_name(col):
    col = str(col).lower().strip()
    col = col.replace("ı", "i").replace("İ", "i")
    col = col.replace("ğ", "g").replace("ü", "u")
    col = col.replace("ş", "s").replace("ö", "o").replace("ç", "c")
    col = col.replace("–", "-").replace("—", "-")
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")
    return col

# Kolon adlarını temizlenmiş hale getiriyoruz
rename_map = {col: clean_column_name(col) for col in building_risk_df.columns}
building_risk_df = building_risk_df.rename(columns=rename_map)

print("Temizlenmiş kolonlar:")
for col in building_risk_df.columns:
    print(col)

# Mahalle adını eşleştirme için standartlaştırıyoruz
building_risk_df["mahalle_key"] = building_risk_df["mahalle_adi"].apply(normalize_text)

# Yaş kolonlarını doğrudan tanımlıyoruz
old_col = "1980_oncesi"
mid_col = "1980_2000_arasi"
new_col = "2000_sonrasi"

# Gerekli yaş kolonları var mı kontrol ediyoruz
required_age_cols = [old_col, mid_col, new_col]
missing_age_cols = [col for col in required_age_cols if col not in building_risk_df.columns]

if len(missing_age_cols) > 0:
    raise ValueError(f"Eksik yaş kolonları var: {missing_age_cols}")

# Kat kolonlarını buluyoruz
# Burada özellikle 'kat' geçen kolonları alıyoruz.
all_floor_cols = [
    col for col in building_risk_df.columns
    if "kat" in col
]

# Düşük kat kolonlarını ayırıyoruz.
# 1-4 kat arası bina sayısı yüksek kat riskine dahil edilmeyecek.
low_floor_cols = [
    col for col in all_floor_cols
    if (
        col.startswith("1_4")
        or col.startswith("1_4_kat")
        or "1_4_kat" in col
    )
]

# Yüksek kat kolonları:
# Kat kolonları içinde düşük kat olmayan her şeyi yüksek kat etkisine dahil ediyoruz.
# Böylece veri setinde varsa 5-9, 10-19, 20+ kolonlarının tamamı alınır.
high_floor_cols = [
    col for col in all_floor_cols
    if col not in low_floor_cols
]

print("\nYaş kolonları:")
print("1980 öncesi:", old_col)
print("1980-2000 arası:", mid_col)
print("2000 sonrası:", new_col)

print("\nTüm kat kolonları:")
print(all_floor_cols)

print("\nDüşük kat hesabı dışında bırakılan kolonlar:")
print(low_floor_cols)

print("\nYüksek kat hesabında kullanılan kolonlar:")
print(high_floor_cols)

# Eğer yüksek kat kolonları bulunamazsa hata veriyoruz
if len(high_floor_cols) == 0:
    raise ValueError("Yüksek kat kolonları bulunamadı. Kolon adlarını kontrol etmek gerekiyor.")

# Sayısal kolonları güvenli şekilde sayıya çeviriyoruz
numeric_cols = required_age_cols + all_floor_cols

for col in numeric_cols:
    building_risk_df[col] = pd.to_numeric(
        building_risk_df[col],
        errors="coerce"
    ).fillna(0)

# Yaş grubu bina sayıları
building_risk_df["old_building_count"] = building_risk_df[old_col]
building_risk_df["mid_building_count"] = building_risk_df[mid_col]
building_risk_df["new_building_count"] = building_risk_df[new_col]

# Yaş gruplarına göre toplam bina sayısı
building_risk_df["total_building_count"] = (
    building_risk_df["old_building_count"] +
    building_risk_df["mid_building_count"] +
    building_risk_df["new_building_count"]
)

# Yüksek katlı bina sayısı
building_risk_df["high_floor_building_count"] = (
    building_risk_df[high_floor_cols]
    .sum(axis=1)
)

# Sıfıra bölme hatasını engelliyoruz
building_risk_df["total_building_count"] = building_risk_df["total_building_count"].replace(0, np.nan)

# Oranlar
building_risk_df["old_building_ratio"] = (
    building_risk_df["old_building_count"] / building_risk_df["total_building_count"]
).fillna(0)

building_risk_df["mid_building_ratio"] = (
    building_risk_df["mid_building_count"] / building_risk_df["total_building_count"]
).fillna(0)

building_risk_df["new_building_ratio"] = (
    building_risk_df["new_building_count"] / building_risk_df["total_building_count"]
).fillna(0)

building_risk_df["high_floor_ratio"] = (
    building_risk_df["high_floor_building_count"] / building_risk_df["total_building_count"]
).fillna(0)

# Normalize fonksiyonu
def minmax(series):
    if series.max() == series.min():
        return series * 0
    return (series - series.min()) / (series.max() - series.min())

# Risk bileşenlerini normalize ediyoruz
building_risk_df["old_building_risk_norm"] = minmax(building_risk_df["old_building_ratio"])
building_risk_df["mid_building_risk_norm"] = minmax(building_risk_df["mid_building_ratio"])
building_risk_df["high_floor_risk_norm"] = minmax(building_risk_df["high_floor_ratio"])

# Bina kaynaklı yol kapanma riski
# Eski bina oranı en yüksek ağırlıkta.
# 1980-2000 arası bina oranı ikinci sırada.
# Yüksek kat oranı üçüncü bileşen.
building_risk_df["building_impact_risk"] = (
    0.50 * building_risk_df["old_building_risk_norm"] +
    0.30 * building_risk_df["mid_building_risk_norm"] +
    0.20 * building_risk_df["high_floor_risk_norm"]
)

# Risk seviyesi etiketi
def risk_label(value):
    if value >= 0.66:
        return "Yüksek"
    elif value >= 0.33:
        return "Orta"
    else:
        return "Düşük"

building_risk_df["building_risk_level"] = building_risk_df["building_impact_risk"].apply(risk_label)

# Kontrol 1:
# high_floor_building_count toplam bina sayısından büyük mü?
building_risk_df["high_floor_check"] = (
    building_risk_df["high_floor_building_count"] <= building_risk_df["total_building_count"]
)

if building_risk_df["high_floor_check"].all():
    print("\nKontrol başarılı: high_floor_building_count toplam bina sayısını aşmıyor.")
else:
    print("\nUYARI: Bazı mahallelerde high_floor_building_count toplam bina sayısını aşıyor. Kat kolonları tekrar kontrol edilmeli.")

# Kontrol 2:
# Yaş oranlarının toplamı yaklaşık 1 olmalı
building_risk_df["age_ratio_sum"] = (
    building_risk_df["old_building_ratio"] +
    building_risk_df["mid_building_ratio"] +
    building_risk_df["new_building_ratio"]
)

print("\nYaş oranı toplamı kontrolü:")
display(
    building_risk_df[
        [
            "mahalle_adi",
            "old_building_ratio",
            "mid_building_ratio",
            "new_building_ratio",
            "age_ratio_sum"
        ]
    ]
)

# Son tablo
building_risk_df = building_risk_df.sort_values(
    "building_impact_risk",
    ascending=False
)

display(
    building_risk_df[
        [
            "mahalle_adi",
            "old_building_count",
            "mid_building_count",
            "new_building_count",
            "high_floor_building_count",
            "total_building_count",
            "old_building_ratio",
            "mid_building_ratio",
            "new_building_ratio",
            "high_floor_ratio",
            "building_impact_risk",
            "building_risk_level"
        ]
    ]
)

In [ ]:
# Road network extraction from OpenStreetMap
# ==============================
# HÜCRE 8 - AVCILAR SINIRI VE TÜM ARAÇ YOL AĞI
# ==============================

import time
import requests
import geopandas as gpd
import osmnx as ox
import networkx as nx
import folium
from shapely.geometry import shape

def normalize_search_text(text):
    text = str(text).upper()
    text = text.replace("İ", "I").replace("İ", "I")
    text = text.replace("Ş", "S").replace("Ğ", "G")
    text = text.replace("Ü", "U").replace("Ö", "O").replace("Ç", "C")
    text = text.replace("ı", "I").replace("ş", "S").replace("ğ", "G")
    text = text.replace("ü", "U").replace("ö", "O").replace("ç", "C")
    return text

def get_avcilar_boundary():
    queries = [
        "Avcılar, Istanbul, Turkey",
        "Avcılar, İstanbul, Türkiye",
        "Avcılar İlçesi, İstanbul",
        "Avcılar district, Istanbul",
        "Avcılar Belediyesi, İstanbul"
    ]

    all_candidates = []

    for query in queries:
        print("Denenen sorgu:", query)

        params = {
            "q": query,
            "format": "jsonv2",
            "polygon_geojson": 1,
            "limit": 20,
            "accept-language": "tr"
        }

        headers = {
            "User-Agent": "avcilar-earthquake-road-network-analysis"
        }

        response = requests.get(
            "https://nominatim.openstreetmap.org/search",
            params=params,
            headers=headers
        )

        time.sleep(1)

        if response.status_code != 200:
            print("Nominatim hata kodu:", response.status_code)
            continue

        results = response.json()

        for result in results:
            display_name = result.get("display_name", "")
            geojson_data = result.get("geojson", None)
            geo_type = geojson_data.get("type") if geojson_data else None

            all_candidates.append({
                "query": query,
                "display_name": display_name,
                "class": result.get("class", ""),
                "type": result.get("type", ""),
                "geojson_type": geo_type,
                "osm_type": result.get("osm_type", ""),
                "osm_id": result.get("osm_id", "")
            })

            if geojson_data is None:
                continue

            if geo_type not in ["Polygon", "MultiPolygon"]:
                continue

            normalized_name = normalize_search_text(display_name)

            if "AVCILAR" in normalized_name and "ISTANBUL" in normalized_name:
                geom = shape(geojson_data)

                boundary = gpd.GeoDataFrame(
                    [{
                        "display_name": display_name,
                        "class": result.get("class", ""),
                        "type": result.get("type", ""),
                        "osm_type": result.get("osm_type", ""),
                        "osm_id": result.get("osm_id", "")
                    }],
                    geometry=[geom],
                    crs="EPSG:4326"
                )

                print("\nUygun Avcılar sınırı bulundu:")
                print(display_name)
                return boundary

    print("\nUygun sınır bulunamadı. Gelen adaylar:")
    candidates_df = pd.DataFrame(all_candidates)
    display(candidates_df)

    raise ValueError("Avcılar için uygun Polygon/MultiPolygon sınırı bulunamadı.")

boundary = get_avcilar_boundary()

avcilar_polygon = boundary.geometry.iloc[0]

G = ox.graph_from_polygon(
    avcilar_polygon,
    network_type="drive",
    simplify=True,
    retain_all=True,
    truncate_by_edge=True
)

print("\nAvcılar tüm araç yol ağı çekildi.")
print("Node sayısı:", len(G.nodes))
print("Edge sayısı:", len(G.edges))

display(boundary[["display_name", "class", "type", "osm_type", "osm_id", "geometry"]])

In [ ]:
# ==============================
# HÜCRE 9 - YOL AĞINI TABLOYA ÇEVİRME VE TEMİZLEME
# ==============================

nodes, edges = ox.graph_to_gdfs(G)

def clean_road_name(name_value):
    if isinstance(name_value, list):
        return " / ".join([str(x) for x in name_value])
    if pd.isna(name_value):
        return "İsimsiz yol"
    return str(name_value)

def clean_highway_type(highway_value):
    if isinstance(highway_value, list):
        return " / ".join([str(x) for x in highway_value])
    if pd.isna(highway_value):
        return "unknown"
    return str(highway_value)

edges_clean = edges.copy()

edges_clean["road_name"] = edges_clean["name"].apply(clean_road_name)
edges_clean["highway_clean"] = edges_clean["highway"].apply(clean_highway_type)
edges_clean["length_m"] = edges_clean["length"]

edges_clean["u"] = edges_clean.index.get_level_values(0)
edges_clean["v"] = edges_clean.index.get_level_values(1)
edges_clean["key"] = edges_clean.index.get_level_values(2)

print("Node tablosu:", nodes.shape)
print("Edge tablosu:", edges_clean.shape)

display(edges_clean[["road_name", "highway_clean", "length_m"]].head(20))

In [ ]:
# Road-type vulnerability assignment
# ==============================
# HÜCRE 10 - YOL TİPİ KIRILGANLIĞI
# ==============================

def road_type_vulnerability(highway_value):
    value = str(highway_value).lower()

    if "motorway" in value or "trunk" in value:
        return 0.20
    elif "primary" in value:
        return 0.30
    elif "secondary" in value:
        return 0.40
    elif "tertiary" in value:
        return 0.50
    elif "residential" in value:
        return 0.75
    elif "service" in value or "living_street" in value:
        return 0.85
    else:
        return 0.60

edges_clean["road_type_vulnerability"] = edges_clean["highway_clean"].apply(road_type_vulnerability)

road_type_summary = edges_clean["highway_clean"].value_counts().reset_index()
road_type_summary.columns = ["highway_type", "segment_count"]

display(road_type_summary.head(30))

In [ ]:
# ==============================
# HÜCRE 11 - AVCILAR TÜM ARAÇ YOL AĞI KONTROL HARİTASI
# ==============================

center = boundary.geometry.iloc[0].centroid
center_lat = center.y
center_lon = center.x

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles="cartodbpositron"
)

folium.GeoJson(
    boundary,
    name="Avcılar sınırı",
    style_function=lambda feature: {
        "color": "blue",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m)

folium.GeoJson(
    edges_clean,
    name="Avcılar tüm araç yol ağı",
    style_function=lambda feature: {
        "color": "red",
        "weight": 2,
        "opacity": 0.45
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["road_name", "highway_clean", "length_m"],
        aliases=["Yol adı:", "Yol tipi:", "Uzunluk (m):"]
    )
).add_to(m)

folium.LayerControl().add_to(m)

m

In [ ]:
# Baseline network centrality calculation
# ==============================
# HÜCRE 12 - NORMAL DURUM CENTRALITY
# ==============================

G_undirected = G.to_undirected()

node_centrality_before = nx.betweenness_centrality(
    G_undirected,
    k=min(500, len(G_undirected.nodes)),
    weight="length",
    seed=42
)

edges_scored = edges_clean.copy()

edges_scored["u_centrality_before"] = edges_scored["u"].map(node_centrality_before)
edges_scored["v_centrality_before"] = edges_scored["v"].map(node_centrality_before)

edges_scored["centrality_before"] = (
    edges_scored["u_centrality_before"] + edges_scored["v_centrality_before"]
) / 2

display(
    edges_scored[
        ["road_name", "highway_clean", "length_m", "road_type_vulnerability", "centrality_before"]
    ].sort_values("centrality_before", ascending=False).head(20)
)

In [ ]:
# Integration of neighborhood-level building risk with road segments
# ==============================
# HÜCRE 13 - AVCILAR MAHALLE SINIRLARINI BULMA
# ==============================

def geocode_neighborhood_polygon(mahalle_name):
    queries = [
        f"{mahalle_name} Mahallesi, Avcılar, İstanbul, Türkiye",
        f"{mahalle_name}, Avcılar, İstanbul, Türkiye",
        f"{mahalle_name} Mahallesi, Avcılar, Istanbul, Turkey"
    ]

    headers = {
        "User-Agent": "avcilar-neighborhood-geocoder"
    }

    for query in queries:
        params = {
            "q": query,
            "format": "jsonv2",
            "polygon_geojson": 1,
            "limit": 5,
            "accept-language": "tr"
        }

        response = requests.get(
            "https://nominatim.openstreetmap.org/search",
            params=params,
            headers=headers
        )

        time.sleep(1)

        if response.status_code != 200:
            continue

        results = response.json()

        for result in results:
            display_name = result.get("display_name", "")
            geojson_data = result.get("geojson", None)

            if geojson_data is None:
                continue

            if geojson_data.get("type") in ["Polygon", "MultiPolygon"]:
                if "Avcılar" in display_name or "Istanbul" in display_name or "İstanbul" in display_name:
                    return shape(geojson_data), display_name

    return None, None

neighborhood_rows = []

for mahalle in building_risk_df["mahalle_adi"].unique():
    print("Mahalle aranıyor:", mahalle)
    geom, display_name = geocode_neighborhood_polygon(mahalle)

    if geom is not None:
        neighborhood_rows.append({
            "mahalle_adi": mahalle,
            "mahalle_key": normalize_text(mahalle),
            "display_name": display_name,
            "geometry": geom
        })
        print("Bulundu:", display_name)
    else:
        print("Bulunamadı:", mahalle)

neighborhoods_gdf = gpd.GeoDataFrame(
    neighborhood_rows,
    geometry="geometry",
    crs="EPSG:4326"
)

print("Bulunan mahalle sınırı sayısı:", len(neighborhoods_gdf))
display(neighborhoods_gdf[["mahalle_adi", "display_name"]])

In [ ]:
# ==============================
# HÜCRE 14 - MAHALLE BİNA RİSKİNİ YOLLARA BAĞLAMA
# ==============================

neighborhoods_risk_gdf = neighborhoods_gdf.merge(
    building_risk_df[
        [
            "mahalle_key",
            "building_impact_risk",
            "building_risk_level",
            "old_building_ratio",
            "mid_building_ratio",
            "high_floor_ratio"
        ]
    ],
    on="mahalle_key",
    how="left"
)

edges_for_join = edges_scored.copy()
edges_for_join["edge_id"] = range(len(edges_for_join))

edges_projected = edges_for_join.to_crs(epsg=3857)

edge_points_projected = edges_projected.copy()
edge_points_projected["geometry"] = edges_projected.geometry.centroid

edge_points = edge_points_projected.to_crs(epsg=4326)

joined = gpd.sjoin(
    edge_points[["edge_id", "geometry"]],
    neighborhoods_risk_gdf[
        [
            "mahalle_adi",
            "mahalle_key",
            "building_impact_risk",
            "building_risk_level",
            "old_building_ratio",
            "mid_building_ratio",
            "high_floor_ratio",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

joined = joined.drop_duplicates(subset=["edge_id"])

edges_model = edges_for_join.merge(
    joined.drop(columns=["geometry", "index_right"], errors="ignore"),
    on="edge_id",
    how="left"
)

median_building_risk = building_risk_df["building_impact_risk"].median()

edges_model["building_impact_risk"] = edges_model["building_impact_risk"].fillna(median_building_risk)
edges_model["building_risk_level"] = edges_model["building_risk_level"].fillna("Bilinmiyor/Orta")
edges_model["old_building_ratio"] = edges_model["old_building_ratio"].fillna(building_risk_df["old_building_ratio"].median())
edges_model["mid_building_ratio"] = edges_model["mid_building_ratio"].fillna(building_risk_df["mid_building_ratio"].median())
edges_model["high_floor_ratio"] = edges_model["high_floor_ratio"].fillna(building_risk_df["high_floor_ratio"].median())

print("Yol segmenti sayısı:", len(edges_model))
print("Mahalle bilgisi eşleşmeyen segment sayısı:", edges_model["mahalle_adi"].isna().sum())

display(
    edges_model[
        [
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "road_type_vulnerability",
            "building_impact_risk",
            "building_risk_level",
            "centrality_before"
        ]
    ].head(20)
)

In [ ]:
# T0 immediate post-earthquake closure scenario
# ==============================
# HÜCRE 15 - AĞIR DEPREM KAPANMA RİSKİ
# ==============================

edges_model["centrality_norm"] = minmax(edges_model["centrality_before"])
edges_model["length_norm"] = minmax(edges_model["length_m"].clip(upper=500))

edges_model["earthquake_closure_risk"] = (
    0.50 * edges_model["building_impact_risk"] +
    0.35 * edges_model["road_type_vulnerability"] +
    0.15 * edges_model["length_norm"]
)

edges_model["closure_probability_T0"] = (
    0.08 + 0.72 * edges_model["earthquake_closure_risk"]
).clip(0.05, 0.85)

def cleanup_reduction_by_road_type(highway_value):
    value = str(highway_value).lower()

    if "motorway" in value or "trunk" in value:
        return 0.75
    elif "primary" in value:
        return 0.60
    elif "secondary" in value:
        return 0.45
    elif "tertiary" in value:
        return 0.25
    elif "residential" in value:
        return 0.10
    elif "service" in value or "living_street" in value:
        return 0.05
    else:
        return 0.15

edges_model["cleanup_reduction"] = edges_model["highway_clean"].apply(cleanup_reduction_by_road_type)

edges_model["closure_probability_T1"] = (
    edges_model["closure_probability_T0"] * (1 - edges_model["cleanup_reduction"])
).clip(0.01, 0.80)

display(
    edges_model[
        [
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "building_impact_risk",
            "road_type_vulnerability",
            "earthquake_closure_risk",
            "closure_probability_T0",
            "closure_probability_T1"
        ]
    ].sort_values("closure_probability_T0", ascending=False).head(30)
)

In [ ]:
# ==============================
# HÜCRE 16 - AĞIR DEPREM KAPANMA SENARYOSU
# FINAL İÇİN DETERMINISTIK RİSK EŞİĞİ
# ==============================

# T0 için en yüksek riskli %20 yol kapalı kabul edilir
T0_threshold = edges_model["closure_probability_T0"].quantile(0.80)

# T1 için ilk 4-5 saat sonrası daha az yol kapalı kalacağı varsayılır
# Bu yüzden en yüksek riskli %15 yol kapalı kabul edilir
T1_threshold = edges_model["closure_probability_T1"].quantile(0.85)

edges_model["closed_T0"] = edges_model["closure_probability_T0"] >= T0_threshold
edges_model["closed_T1"] = edges_model["closure_probability_T1"] >= T1_threshold

print("T0 kapanma eşiği:", T0_threshold)
print("T1 kapanma eşiği:", T1_threshold)

print("T0 kapalı segment sayısı:", edges_model["closed_T0"].sum())
print("T1 kapalı segment sayısı:", edges_model["closed_T1"].sum())

print("T0 kapalı toplam uzunluk km:", round(edges_model.loc[edges_model["closed_T0"], "length_m"].sum() / 1000, 2))
print("T1 kapalı toplam uzunluk km:", round(edges_model.loc[edges_model["closed_T1"], "length_m"].sum() / 1000, 2))

display(
    edges_model[
        [
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "closure_probability_T0",
            "closure_probability_T1",
            "closed_T0",
            "closed_T1"
        ]
    ].sort_values("closure_probability_T0", ascending=False).head(30)
)

In [ ]:
# T1 partial recovery and reopening logic
# ==============================
# HÜCRE 16B - T1 AÇILMA/KAPANMA MANTIĞINI GÜÇLENDİRME
# ==============================

# Bu hücre, T1 senaryosunu daha gerçekçi hale getirir.
# Mantık:
# T1'de yeni yol kapanması üretmiyoruz.
# T1 = T0'da kapalı olan yolların bir kısmının ilk 4-5 saat içinde açılmış hali.
# Bu yüzden T1 kapalı yollar, T0 kapalı yolların alt kümesi olmalıdır.

# 1. Yol hiyerarşisi / müdahale erişilebilirliği skoru
# Büyük ve stratejik yolların ilk müdahale sürecinde daha hızlı açılabileceğini varsayıyoruz.

def road_hierarchy_reopening_score(highway_value):
    value = str(highway_value).lower()

    if "motorway" in value or "trunk" in value:
        return 1.00
    elif "primary" in value:
        return 0.85
    elif "secondary" in value:
        return 0.70
    elif "tertiary" in value:
        return 0.50
    elif "residential" in value:
        return 0.25
    elif "service" in value or "living_street" in value:
        return 0.15
    else:
        return 0.30

edges_model["road_hierarchy_reopening_score"] = edges_model["highway_clean"].apply(
    road_hierarchy_reopening_score
)

# 2. Centrality normalizasyonu
# Eğer daha önce centrality_norm oluşmadıysa yeniden oluşturalım.

def minmax_safe(series):
    if series.max() == series.min():
        return series * 0
    return (series - series.min()) / (series.max() - series.min())

if "centrality_norm" not in edges_model.columns:
    edges_model["centrality_norm"] = minmax_safe(edges_model["centrality_before"])

# 3. Bina riskinin tersini alıyoruz.
# Bina riski düşükse, yolun açılma ihtimali daha yüksek varsayılır.

edges_model["low_building_risk_score"] = 1 - edges_model["building_impact_risk"]

# 4. T1 açılma öncelik skoru
# Yol büyüklüğü/hiyerarşisi en yüksek ağırlıkta.
# Sonra network centrality.
# Sonra bina riskinin düşük olması.

edges_model["t1_reopening_priority_score"] = (
    0.50 * edges_model["road_hierarchy_reopening_score"] +
    0.30 * edges_model["centrality_norm"] +
    0.20 * edges_model["low_building_risk_score"]
)

# 5. T1'de kapalı kalma skoru
# Bir yolun T1'de kapalı kalması için:
# - T0 kapanma olasılığı yüksek olmalı
# - Açılma önceliği düşük olmalı
#
# Yani yüksek risk + düşük müdahale önceliği = T1'de kapalı kalma ihtimali yüksek.

edges_model["t1_remaining_closure_score"] = (
    0.60 * edges_model["closure_probability_T0"] +
    0.40 * (1 - edges_model["t1_reopening_priority_score"])
)

# 6. T1 kapalı yolları T0 kapalı yollar içinden seçiyoruz.
# Böylece T1, zamansal olarak T0'ın alt kümesi olur.
# Yani T1'de kapalı olan yol, T0'da da kapalı olmalıdır.

t0_closed_edges = edges_model[edges_model["closed_T0"] == True].copy()

# T0'da kapalı yolların yaklaşık %60'ının T1'de hâlâ kapalı kalacağını varsayıyoruz.
# Bu oranı değiştirebilirsin:
# 0.50 = daha iyimser T1
# 0.60 = orta senaryo
# 0.70 = daha kötümser T1

t1_remaining_share_of_T0 = 0.60

t1_closed_count = int(round(len(t0_closed_edges) * t1_remaining_share_of_T0))

# Eğer T0'da hiç kapalı yol yoksa hata almamak için kontrol
if t1_closed_count > 0:
    t1_closed_edge_ids = (
        t0_closed_edges
        .sort_values("t1_remaining_closure_score", ascending=False)
        .head(t1_closed_count)["edge_id"]
        .tolist()
    )
else:
    t1_closed_edge_ids = []

# closed_T1'i yeni sağlam mantıkla güncelliyoruz
edges_model["closed_T1"] = edges_model["edge_id"].isin(t1_closed_edge_ids)

# T1 kapanma olasılığını da güncel mantıkla yeniden ifade ediyoruz.
# Bu değer artık yolun T1'de kapalı kalma eğilimini temsil eder.

edges_model["closure_probability_T1"] = (
    edges_model["closure_probability_T0"] *
    (1 - 0.75 * edges_model["t1_reopening_priority_score"])
).clip(0.01, 0.80)

# 7. T1'de açılan yollar
# T0'da kapalı olup T1'de açık hale gelen yollar

edges_model["reopened_T1"] = (
    (edges_model["closed_T0"] == True) &
    (edges_model["closed_T1"] == False)
)

# 8. Zamansal tutarlılık kontrolü
# T1'de kapalı olup T0'da kapalı olmayan yol olmamalı.

temporal_inconsistency = edges_model[
    (edges_model["closed_T1"] == True) &
    (edges_model["closed_T0"] == False)
]

print("T0 kapalı segment sayısı:", edges_model["closed_T0"].sum())
print("T1 kapalı segment sayısı:", edges_model["closed_T1"].sum())
print("T1'de açılan segment sayısı:", edges_model["reopened_T1"].sum())
print("T1'de kapalı ama T0'da kapalı olmayan segment sayısı:", len(temporal_inconsistency))

print("\nT1 kapalı toplam uzunluk km:", round(edges_model.loc[edges_model["closed_T1"], "length_m"].sum() / 1000, 2))
print("T1'de açılan toplam uzunluk km:", round(edges_model.loc[edges_model["reopened_T1"], "length_m"].sum() / 1000, 2))

# 9. T1'de açılan yolların kontrol tablosu
print("\nT1'de açılan yollar - ilk 30 segment:")
display(
    edges_model[
        edges_model["reopened_T1"] == True
    ][
        [
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "closure_probability_T0",
            "closure_probability_T1",
            "road_hierarchy_reopening_score",
            "centrality_norm",
            "building_impact_risk",
            "t1_reopening_priority_score",
            "t1_remaining_closure_score",
            "closed_T0",
            "closed_T1",
            "reopened_T1"
        ]
    ].sort_values("t1_reopening_priority_score", ascending=False).head(30)
)

# 10. T1'de kapalı kalan yolların kontrol tablosu
print("\nT1'de kapalı kalan yollar - ilk 30 segment:")
display(
    edges_model[
        edges_model["closed_T1"] == True
    ][
        [
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "closure_probability_T0",
            "closure_probability_T1",
            "road_hierarchy_reopening_score",
            "centrality_norm",
            "building_impact_risk",
            "t1_reopening_priority_score",
            "t1_remaining_closure_score",
            "closed_T0",
            "closed_T1"
        ]
    ].sort_values("t1_remaining_closure_score", ascending=False).head(30)
)

# 11. Yol tipi bazında T1 özeti
print("\nYol tipi bazında T1 özeti:")
t1_type_summary = edges_model.groupby("highway_clean").agg(
    total_segment=("edge_id", "count"),
    closed_T0_count=("closed_T0", "sum"),
    closed_T1_count=("closed_T1", "sum"),
    reopened_T1_count=("reopened_T1", "sum"),
    avg_reopening_priority=("t1_reopening_priority_score", "mean"),
    avg_remaining_closure=("t1_remaining_closure_score", "mean")
).reset_index()

display(
    t1_type_summary.sort_values("closed_T1_count", ascending=False).head(30)
)

In [ ]:
# Network degradation and stress analysis
# ==============================
# HÜCRE 17 - KAPALI YOLLARI NETWORKTEN ÇIKARMA
# ==============================

def remove_closed_edges(G_original, edges_df, closed_col):
    G_new = G_original.copy()
    removed_edges = []

    closed_edges = edges_df[edges_df[closed_col]].copy()

    for _, row in closed_edges.iterrows():
        u = row["u"]
        v = row["v"]
        key = row["key"]

        if G_new.has_edge(u, v, key):
            G_new.remove_edge(u, v, key)
            removed_edges.append((u, v, key))

    return G_new, removed_edges

G_T0, removed_T0 = remove_closed_edges(G, edges_model, "closed_T0")
G_T1, removed_T1 = remove_closed_edges(G, edges_model, "closed_T1")

print("T0 kaldırılan edge sayısı:", len(removed_T0))
print("T1 kaldırılan edge sayısı:", len(removed_T1))

In [ ]:
# ==============================
# HÜCRE 18 - NETWORK BOZULMA METRİKLERİ
# ==============================

def network_basic_metrics(G_input):
    G_temp = G_input.to_undirected()
    components = list(nx.connected_components(G_temp))
    largest_component = max(components, key=len)

    return {
        "node_count": len(G_input.nodes),
        "edge_count": len(G_input.edges),
        "component_count": len(components),
        "largest_component_nodes": len(largest_component),
        "largest_component_ratio": len(largest_component) / len(G_input.nodes),
        "nodes_outside_largest_component": len(G_input.nodes) - len(largest_component)
    }

normal_metrics = network_basic_metrics(G)
T0_metrics = network_basic_metrics(G_T0)
T1_metrics = network_basic_metrics(G_T1)

network_comparison = pd.DataFrame([
    {"scenario": "Normal durum", **normal_metrics},
    {"scenario": "T0 - Deprem sonrası ilk etki", **T0_metrics},
    {"scenario": "T1 - İlk 4-5 saat sonrası", **T1_metrics}
])

network_comparison

In [ ]:
# ==============================
# HÜCRE 19 - SENARYO SONRASI STRES ANALİZİ
# ==============================

def calculate_stress_after_scenario(G_scenario, edges_model_before, scenario_name):
    # Senaryo sonrası graph'ı undirected hale getiriyoruz
    G_scenario_undirected = G_scenario.to_undirected()

    # Senaryo sonrası node centrality hesaplıyoruz
    node_centrality_after = nx.betweenness_centrality(
        G_scenario_undirected,
        k=min(500, len(G_scenario_undirected.nodes)),
        weight="length",
        seed=42
    )

    # Senaryo sonrası graph'ı tabloya çeviriyoruz
    nodes_after, edges_after = ox.graph_to_gdfs(G_scenario)

    # ÖNEMLİ:
    # ox.graph_to_gdfs çıktısında u, v, key index olarak geliyor.
    # Bunları normal kolona çeviriyoruz ve index karmaşasını kaldırıyoruz.
    edges_after = edges_after.reset_index()

    # Senaryo sonrası centrality değerlerini edge'lere bağlıyoruz
    edges_after["u_centrality_after"] = edges_after["u"].map(node_centrality_after)
    edges_after["v_centrality_after"] = edges_after["v"].map(node_centrality_after)

    edges_after["centrality_after"] = (
        edges_after["u_centrality_after"] + edges_after["v_centrality_after"]
    ) / 2

    # Senaryo öncesi bilgileri alıyoruz
    # Burada da index'i sıfırlıyoruz ki u/v/key hem index hem kolon olmasın.
    before_cols = edges_model_before[
        [
            "u", "v", "key",
            "road_name",
            "highway_clean",
            "length_m",
            "mahalle_adi",
            "centrality_before",
            "earthquake_closure_risk"
        ]
    ].copy().reset_index(drop=True)

    # Önceki ve sonraki durumu aynı yol segmentleri üzerinden birleştiriyoruz
    stress_edges = edges_after.merge(
        before_cols,
        on=["u", "v", "key"],
        how="left"
    )

    # Eksik centrality_before değerlerini 0 yapıyoruz
    stress_edges["centrality_before"] = stress_edges["centrality_before"].fillna(0)

    # Centrality artışı = senaryo sonrası - normal durum
    stress_edges["centrality_increase"] = (
        stress_edges["centrality_after"] - stress_edges["centrality_before"]
    )

    # Sadece centrality değeri artan açık yolları alıyoruz
    stressed_edges = stress_edges[
        stress_edges["centrality_increase"] > 0
    ].copy()

    # Yol/cadde adı bazında özet tablo oluşturuyoruz
    stress_road_summary = stressed_edges.groupby("road_name").agg(
        stressed_segment_count=("centrality_increase", "count"),
        avg_increase=("centrality_increase", "mean"),
        max_increase=("centrality_increase", "max"),
        total_stressed_length_km=("length_m", lambda x: x.sum() / 1000),
        avg_closure_risk=("earthquake_closure_risk", "mean")
    ).reset_index()

    # İsimsiz yolları final yorumdan çıkarıyoruz
    stress_road_summary = stress_road_summary[
        stress_road_summary["road_name"] != "İsimsiz yol"
    ].copy()

    stress_road_summary["scenario"] = scenario_name

    stress_road_summary = stress_road_summary.sort_values(
        "max_increase",
        ascending=False
    )

    return stressed_edges, stress_road_summary


stressed_edges_T0, stress_road_summary_T0 = calculate_stress_after_scenario(
    G_T0,
    edges_model,
    "T0 - Deprem sonrası ilk etki"
)

stressed_edges_T1, stress_road_summary_T1 = calculate_stress_after_scenario(
    G_T1,
    edges_model,
    "T1 - İlk 4-5 saat sonrası"
)

print("T0 stres yolları:")
display(stress_road_summary_T0.head(20))

print("T1 stres yolları:")
display(stress_road_summary_T1.head(20))

In [ ]:
# Identification of stress corridors, local bottlenecks, and critical open backbone roads
# ==============================
# HÜCRE 20 - ANLAMLI STRES YOLLARI VE YEREL DARBOĞAZLAR
# ==============================

def classify_stress_roads(stress_summary):
    robust_stress = stress_summary[
        (stress_summary["stressed_segment_count"] >= 3) &
        (stress_summary["total_stressed_length_km"] >= 0.3) &
        (stress_summary["max_increase"] >= stress_summary["max_increase"].quantile(0.75))
    ].copy()

    local_bottlenecks = stress_summary[
        (stress_summary["total_stressed_length_km"] < 0.3) &
        (stress_summary["max_increase"] >= stress_summary["max_increase"].quantile(0.80))
    ].copy()

    return robust_stress, local_bottlenecks

robust_stress_T0, local_bottlenecks_T0 = classify_stress_roads(stress_road_summary_T0)
robust_stress_T1, local_bottlenecks_T1 = classify_stress_roads(stress_road_summary_T1)

print("T0 robust stres yolu sayısı:", len(robust_stress_T0))
display(robust_stress_T0.head(20))

print("T0 yerel darboğaz adayı sayısı:", len(local_bottlenecks_T0))
display(local_bottlenecks_T0.head(20))

print("T1 robust stres yolu sayısı:", len(robust_stress_T1))
display(robust_stress_T1.head(20))

print("T1 yerel darboğaz adayı sayısı:", len(local_bottlenecks_T1))
display(local_bottlenecks_T1.head(20))

In [ ]:
# Interactive map generation
# ==============================
# HÜCRE 21 - HARİTA VE RAPOR HAZIRLIĞI
# ==============================

from branca.element import Element
from IPython.display import Markdown, display

# GeoDataFrame güvenliği
edges_model = gpd.GeoDataFrame(edges_model, geometry="geometry", crs=edges_scored.crs)
stressed_edges_T0 = gpd.GeoDataFrame(stressed_edges_T0, geometry="geometry", crs=edges_scored.crs)
stressed_edges_T1 = gpd.GeoDataFrame(stressed_edges_T1, geometry="geometry", crs=edges_scored.crs)

# Harita merkezi
center = boundary.geometry.iloc[0].centroid
center_lat = center.y
center_lon = center.x

def add_title(m, title, subtitle):
    title_html = f"""
    <div style="
        position: fixed;
        top: 10px;
        left: 50px;
        width: 560px;
        z-index: 9999;
        background-color: white;
        border: 2px solid #444;
        border-radius: 8px;
        padding: 10px;
        font-size: 14px;
        box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
    ">
        <b>{title}</b><br>
        <span style="font-size: 12px;">{subtitle}</span>
    </div>
    """
    m.get_root().html.add_child(Element(title_html))

def add_legend(m, items, note=""):
    rows = ""
    for color, label in items:
        rows += f"""
        <div style="margin-bottom: 5px;">
            <span style="
                display: inline-block;
                width: 18px;
                height: 6px;
                background-color: {color};
                margin-right: 8px;
                vertical-align: middle;
            "></span>
            {label}
        </div>
        """

    legend_html = f"""
    <div style="
        position: fixed;
        bottom: 30px;
        left: 50px;
        width: 390px;
        z-index: 9999;
        background-color: white;
        border: 2px solid #444;
        border-radius: 8px;
        padding: 10px;
        font-size: 12px;
        box-shadow: 2px 2px 6px rgba(0,0,0,0.3);
    ">
        <b>Lejant</b><br><br>
        {rows}
        <div style="font-size: 11px; margin-top: 8px; color: #555;">
            {note}
        </div>
    </div>
    """
    m.get_root().html.add_child(Element(legend_html))

# Harita için sayısal kolonları yuvarlayalım
for df in [edges_model, stressed_edges_T0, stressed_edges_T1]:
    for col in df.columns:
        if df[col].dtype in ["float64", "float32"]:
            df[col] = df[col].round(6)

# Yüksek risk eşiği
risk_q80 = edges_model["closure_probability_T0"].quantile(0.80)
risk_q90 = edges_model["closure_probability_T0"].quantile(0.90)

high_risk_edges = edges_model[
    edges_model["closure_probability_T0"] >= risk_q80
].copy()

very_high_risk_edges = edges_model[
    edges_model["closure_probability_T0"] >= risk_q90
].copy()

closed_T0_edges = edges_model[edges_model["closed_T0"]].copy()
closed_T1_edges = edges_model[edges_model["closed_T1"]].copy()

print("Harita hazırlığı tamamlandı.")
print("Yüksek risk eşiği q80:", risk_q80)
print("Çok yüksek risk eşiği q90:", risk_q90)
print("T0 kapalı segment:", len(closed_T0_edges))
print("T1 kapalı segment:", len(closed_T1_edges))

In [ ]:
# ==============================
# HÜCRE 22 - HARİTA 1: DEPREM KAPANMA RİSKİ YÜKSEK YOLLAR
# ==============================

m_risk = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles="cartodbpositron"
)

folium.GeoJson(
    boundary,
    name="Avcılar sınırı",
    style_function=lambda feature: {
        "color": "blue",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m_risk)

folium.GeoJson(
    edges_model,
    name="Tüm araç yol ağı",
    style_function=lambda feature: {
        "color": "lightgray",
        "weight": 1,
        "opacity": 0.30
    }
).add_to(m_risk)

folium.GeoJson(
    high_risk_edges,
    name="Yüksek kapanma riski",
    style_function=lambda feature: {
        "color": "orange",
        "weight": 3,
        "opacity": 0.75
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "building_impact_risk",
            "road_type_vulnerability",
            "earthquake_closure_risk",
            "closure_probability_T0"
        ],
        aliases=[
            "Yol adı:",
            "Yol tipi:",
            "Mahalle:",
            "Bina etkisi riski:",
            "Yol tipi kırılganlığı:",
            "Deprem kapanma riski:",
            "T0 kapanma olasılığı:"
        ]
    )
).add_to(m_risk)

folium.GeoJson(
    very_high_risk_edges,
    name="Çok yüksek kapanma riski",
    style_function=lambda feature: {
        "color": "red",
        "weight": 4,
        "opacity": 0.90
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "building_impact_risk",
            "road_type_vulnerability",
            "earthquake_closure_risk",
            "closure_probability_T0"
        ],
        aliases=[
            "Yol adı:",
            "Yol tipi:",
            "Mahalle:",
            "Bina etkisi riski:",
            "Yol tipi kırılganlığı:",
            "Deprem kapanma riski:",
            "T0 kapanma olasılığı:"
        ]
    )
).add_to(m_risk)

folium.LayerControl().add_to(m_risk)

add_title(
    m_risk,
    "#HARİTA_1 Deprem Kapanma Riski Yüksek Yollar",
    "Bina riski, yol tipi kırılganlığı ve yol uzunluğu üzerinden kapanma olasılığı yüksek çıkan yol segmentleri."
)

add_legend(
    m_risk,
    [
        ("blue", "Avcılar sınırı"),
        ("lightgray", "Tüm araç yol ağı"),
        ("orange", "Yüksek kapanma riski"),
        ("red", "Çok yüksek kapanma riski")
    ],
    "Bu harita kesin kapanacak yolları değil, modele göre kapanma potansiyeli yüksek yolları gösterir."
)

m_risk.save("harita_1_deprem_kapanma_riski_yuksek_yollar.html")

m_risk

In [ ]:
# ==============================
# HÜCRE 23 - HARİTA 2: T0 KAPALI YOLLAR + T0 STRES YOLLARI
# ==============================

top_T0_robust_names = robust_stress_T0.head(20)["road_name"].tolist()
top_T0_bottleneck_names = local_bottlenecks_T0.head(20)["road_name"].tolist()

T0_robust_edges = stressed_edges_T0[
    stressed_edges_T0["road_name"].isin(top_T0_robust_names)
].copy()

T0_bottleneck_edges = stressed_edges_T0[
    stressed_edges_T0["road_name"].isin(top_T0_bottleneck_names)
].copy()

m_T0 = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles="cartodbpositron"
)

folium.GeoJson(
    boundary,
    name="Avcılar sınırı",
    style_function=lambda feature: {
        "color": "blue",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m_T0)

folium.GeoJson(
    edges_model,
    name="Tüm araç yol ağı",
    style_function=lambda feature: {
        "color": "lightgray",
        "weight": 1,
        "opacity": 0.25
    }
).add_to(m_T0)

folium.GeoJson(
    closed_T0_edges,
    name="T0 kapalı yollar",
    style_function=lambda feature: {
        "color": "black",
        "weight": 3,
        "opacity": 0.75
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "closure_probability_T0"
        ],
        aliases=[
            "Kapalı yol:",
            "Yol tipi:",
            "Mahalle:",
            "T0 kapanma olasılığı:"
        ]
    )
).add_to(m_T0)

folium.GeoJson(
    T0_robust_edges,
    name="T0 ana stres koridorları",
    style_function=lambda feature: {
        "color": "orange",
        "weight": 5,
        "opacity": 0.90
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "centrality_before",
            "centrality_after",
            "centrality_increase"
        ],
        aliases=[
            "Stres yolu:",
            "Yol tipi:",
            "Mahalle:",
            "Önceki centrality:",
            "Sonraki centrality:",
            "Centrality artışı:"
        ]
    )
).add_to(m_T0)

folium.GeoJson(
    T0_bottleneck_edges,
    name="T0 yerel darboğaz adayları",
    style_function=lambda feature: {
        "color": "purple",
        "weight": 4,
        "opacity": 0.90
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "centrality_increase"
        ],
        aliases=[
            "Darboğaz adayı:",
            "Yol tipi:",
            "Mahalle:",
            "Centrality artışı:"
        ]
    )
).add_to(m_T0)

folium.LayerControl().add_to(m_T0)

add_title(
    m_T0,
    "#HARİTA_2 T0: Deprem Sonrası İlk Etki",
    "Deprem sonrası ilk anda kapalı varsayılan yollar ve bu kapanmalar sonrası stres altına giren açık yollar."
)

add_legend(
    m_T0,
    [
        ("blue", "Avcılar sınırı"),
        ("lightgray", "Tüm araç yol ağı"),
        ("black", "T0 kapalı yollar"),
        ("orange", "T0 ana stres koridorları"),
        ("purple", "T0 yerel darboğaz adayları")
    ],
    "T0, deprem sonrası ilk etki anını temsil eder."
)

m_T0.save("harita_2_T0_kapali_yollar_ve_stres_yollari.html")

m_T0

In [ ]:
print(edges_model[["closure_probability_T0", "closure_probability_T1", "closed_T0", "closed_T1"]].head())

print("T0 kapalı segment sayısı:", edges_model["closed_T0"].sum())
print("T1 kapalı segment sayısı:", edges_model["closed_T1"].sum())

In [ ]:
# ==============================
# HÜCRE - T1 KAPALI YOLLAR + T1 STRES YOLLARI HARİTASI
# ==============================

# Eğer center değişkenleri yoksa yeniden üretelim
center = boundary.geometry.iloc[0].centroid
center_lat = center.y
center_lon = center.x

# Haritada göstereceğimiz T1 ana stres koridorları ve yerel darboğaz adayları
top_T1_robust_names = robust_stress_T1.head(20)["road_name"].tolist()
top_T1_bottleneck_names = local_bottlenecks_T1.head(20)["road_name"].tolist()

T1_robust_edges = stressed_edges_T1[
    stressed_edges_T1["road_name"].isin(top_T1_robust_names)
].copy()

T1_bottleneck_edges = stressed_edges_T1[
    stressed_edges_T1["road_name"].isin(top_T1_bottleneck_names)
].copy()

# T1'de kapalı kabul edilen yollar
closed_T1_edges = edges_model[
    edges_model["closed_T1"] == True
].copy()

# Harita
m_T1 = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles="cartodbpositron"
)

# Avcılar sınırı
folium.GeoJson(
    boundary,
    name="Avcılar sınırı",
    style_function=lambda feature: {
        "color": "blue",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m_T1)

# Tüm araç yol ağı
folium.GeoJson(
    edges_model,
    name="Tüm araç yol ağı",
    style_function=lambda feature: {
        "color": "lightgray",
        "weight": 1,
        "opacity": 0.25
    }
).add_to(m_T1)

# T1 kapalı yollar
folium.GeoJson(
    closed_T1_edges,
    name="T1 kapalı yollar",
    style_function=lambda feature: {
        "color": "black",
        "weight": 3,
        "opacity": 0.75
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "closure_probability_T1"
        ],
        aliases=[
            "Kapalı yol:",
            "Yol tipi:",
            "Mahalle:",
            "T1 kapanma olasılığı:"
        ]
    )
).add_to(m_T1)

# T1 ana stres koridorları
folium.GeoJson(
    T1_robust_edges,
    name="T1 ana stres koridorları",
    style_function=lambda feature: {
        "color": "orange",
        "weight": 5,
        "opacity": 0.90
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "centrality_before",
            "centrality_after",
            "centrality_increase"
        ],
        aliases=[
            "Stres yolu:",
            "Yol tipi:",
            "Mahalle:",
            "Önceki centrality:",
            "Sonraki centrality:",
            "Centrality artışı:"
        ]
    )
).add_to(m_T1)

# T1 yerel darboğaz adayları
folium.GeoJson(
    T1_bottleneck_edges,
    name="T1 yerel darboğaz adayları",
    style_function=lambda feature: {
        "color": "purple",
        "weight": 4,
        "opacity": 0.90
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "centrality_increase"
        ],
        aliases=[
            "Darboğaz adayı:",
            "Yol tipi:",
            "Mahalle:",
            "Centrality artışı:"
        ]
    )
).add_to(m_T1)

# Eğer add_title ve add_legend fonksiyonları daha önce tanımlandıysa kullan
try:
    add_title(
        m_T1,
        "#HARİTA_3 T1: İlk 4-5 Saat Sonrası",
        "Ana yolların kısmen açılabileceği varsayımı altında T1 kapalı yolları, stres koridorları ve yerel darboğaz adayları."
    )

    add_legend(
        m_T1,
        [
            ("blue", "Avcılar sınırı"),
            ("lightgray", "Tüm araç yol ağı"),
            ("black", "T1 kapalı yollar"),
            ("orange", "T1 ana stres koridorları"),
            ("purple", "T1 yerel darboğaz adayları")
        ],
        "T1, depremden sonraki ilk 4-5 saat içinde ana yolların kısmen temizlenmiş olabileceği varsayımını temsil eder."
    )
except:
    pass

folium.LayerControl().add_to(m_T1)

m_T1.save("harita_T1_kapali_yollar_ve_stres_yollari.html")

m_T1

In [ ]:
# T1'de kapalı olup T0'da kapalı olmayan yol var mı?

newly_closed_T1 = edges_model[
    (edges_model["closed_T1"] == True) &
    (edges_model["closed_T0"] == False)
].copy()

print("T1'de kapalı ama T0'da kapalı olmayan segment sayısı:", len(newly_closed_T1))

display(
    newly_closed_T1[
        [
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "closure_probability_T0",
            "closure_probability_T1",
            "closed_T0",
            "closed_T1"
        ]
    ].head(30)
)

In [ ]:
# ==============================
# HÜCRE - TEMEL KRİTİK YOL OMURGASINI BELİRLEME
# ==============================

# Bu hücrede deprem öncesi normal ağda kritik kabul edeceğimiz yol segmentlerini belirliyoruz.
# Mantık:
# 1) centrality_before yüksek olacak
# 2) yol tipi ana yol omurgasına yakın olacak
# 3) isimsiz yolları dışarıda bırakacağız

def is_core_road(highway_value):
    value = str(highway_value).lower()
    core_types = ["motorway", "trunk", "primary", "secondary", "tertiary"]
    return any(htype in value for htype in core_types)

edges_model["is_core_road"] = edges_model["highway_clean"].apply(is_core_road)

# Normal durumdaki centrality için eşik
baseline_critical_threshold = edges_model["centrality_before"].quantile(0.85)

# Deprem öncesi kritik yol segmentleri
baseline_critical_edges = edges_model[
    (edges_model["centrality_before"] >= baseline_critical_threshold) &
    (edges_model["is_core_road"] == True) &
    (edges_model["road_name"] != "İsimsiz yol")
].copy()

print("Deprem öncesi kritik centrality eşiği:", baseline_critical_threshold)
print("Deprem öncesi kritik yol segmenti sayısı:", len(baseline_critical_edges))

display(
    baseline_critical_edges[
        [
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "centrality_before",
            "closure_probability_T0",
            "closure_probability_T1"
        ]
    ].sort_values("centrality_before", ascending=False).head(30)
)

# Yol adı bazında özet tablo
baseline_critical_road_summary = baseline_critical_edges.groupby("road_name").agg(
    segment_count=("road_name", "count"),
    avg_centrality_before=("centrality_before", "mean"),
    max_centrality_before=("centrality_before", "max"),
    total_length_km=("length_m", lambda x: x.sum() / 1000),
    dominant_highway=("highway_clean", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0])
).reset_index()

baseline_critical_road_summary = baseline_critical_road_summary.sort_values(
    "max_centrality_before",
    ascending=False
)

print("\nDeprem öncesi kritik yollar - yol adı bazında özet:")
display(baseline_critical_road_summary.head(30))

In [ ]:
# ==============================
# HÜCRE - T0'DА GRİ KALAN AMA KRİTİK OLAN AÇIK YOLLAR
# ==============================

# T0'da kapanan yollar
closed_T0_names = set(closed_T0_edges["road_name"].dropna().tolist())

# T0 robust stres koridorları
robust_T0_names = set(robust_stress_T0["road_name"].dropna().tolist())

# T0 yerel darboğaz adayları
bottleneck_T0_names = set(local_bottlenecks_T0["road_name"].dropna().tolist())

# Deprem öncesi kritik yollar
baseline_critical_names = set(baseline_critical_edges["road_name"].dropna().tolist())

# T0'da açık kalan ve özel risk grubuna girmeyen ama başlangıçta kritik olan yollar
critical_open_T0 = edges_model[
    (edges_model["road_name"].isin(baseline_critical_names)) &
    (~edges_model["road_name"].isin(closed_T0_names)) &
    (~edges_model["road_name"].isin(robust_T0_names)) &
    (~edges_model["road_name"].isin(bottleneck_T0_names))
].copy()

print("T0'da açık kalan kritik omurga segment sayısı:", len(critical_open_T0))

display(
    critical_open_T0[
        [
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "centrality_before",
            "closure_probability_T0"
        ]
    ].sort_values("centrality_before", ascending=False).head(30)
)

In [ ]:
# ==============================
# HÜCRE - T1'DЕ GRİ KALAN AMA KRİTİK OLAN AÇIK YOLLAR
# ==============================

# T1'de kapanan yollar
closed_T1_names = set(closed_T1_edges["road_name"].dropna().tolist())

# T1 robust stres koridorları
robust_T1_names = set(robust_stress_T1["road_name"].dropna().tolist())

# T1 yerel darboğaz adayları
bottleneck_T1_names = set(local_bottlenecks_T1["road_name"].dropna().tolist())

# T1'de açık kalan ve özel risk grubuna girmeyen ama başlangıçta kritik olan yollar
critical_open_T1 = edges_model[
    (edges_model["road_name"].isin(baseline_critical_names)) &
    (~edges_model["road_name"].isin(closed_T1_names)) &
    (~edges_model["road_name"].isin(robust_T1_names)) &
    (~edges_model["road_name"].isin(bottleneck_T1_names))
].copy()

print("T1'de açık kalan kritik omurga segment sayısı:", len(critical_open_T1))

display(
    critical_open_T1[
        [
            "road_name",
            "highway_clean",
            "mahalle_adi",
            "centrality_before",
            "closure_probability_T1"
        ]
    ].sort_values("centrality_before", ascending=False).head(30)
)

In [ ]:
# Interactive map generation
# ==============================
# HÜCRE - NİHAİ T0 HARİTASI
# ==============================

center = boundary.geometry.iloc[0].centroid
center_lat = center.y
center_lon = center.x

top_T0_robust_names = robust_stress_T0.head(20)["road_name"].tolist()
top_T0_bottleneck_names = local_bottlenecks_T0.head(20)["road_name"].tolist()

T0_robust_edges = stressed_edges_T0[
    stressed_edges_T0["road_name"].isin(top_T0_robust_names)
].copy()

T0_bottleneck_edges = stressed_edges_T0[
    stressed_edges_T0["road_name"].isin(top_T0_bottleneck_names)
].copy()

m_T0_final = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles="cartodbpositron"
)

# Sınır
folium.GeoJson(
    boundary,
    name="Avcılar sınırı",
    style_function=lambda feature: {
        "color": "blue",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m_T0_final)

# Tüm yol ağı
folium.GeoJson(
    edges_model,
    name="Tüm araç yol ağı",
    style_function=lambda feature: {
        "color": "lightgray",
        "weight": 1,
        "opacity": 0.20
    }
).add_to(m_T0_final)

# T0 kapalı yollar
folium.GeoJson(
    closed_T0_edges,
    name="T0 kapalı yollar",
    style_function=lambda feature: {
        "color": "black",
        "weight": 3,
        "opacity": 0.80
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["road_name", "highway_clean", "mahalle_adi", "closure_probability_T0"],
        aliases=["Kapalı yol:", "Yol tipi:", "Mahalle:", "T0 kapanma olasılığı:"]
    )
).add_to(m_T0_final)

# T0 stres koridorları
folium.GeoJson(
    T0_robust_edges,
    name="T0 ana stres koridorları",
    style_function=lambda feature: {
        "color": "orange",
        "weight": 5,
        "opacity": 0.90
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["road_name", "highway_clean", "mahalle_adi", "centrality_increase"],
        aliases=["Stres yolu:", "Yol tipi:", "Mahalle:", "Centrality artışı:"]
    )
).add_to(m_T0_final)

# T0 yerel darboğazlar
folium.GeoJson(
    T0_bottleneck_edges,
    name="T0 yerel darboğaz adayları",
    style_function=lambda feature: {
        "color": "purple",
        "weight": 4,
        "opacity": 0.90
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["road_name", "highway_clean", "mahalle_adi", "centrality_increase"],
        aliases=["Darboğaz adayı:", "Yol tipi:", "Mahalle:", "Centrality artışı:"]
    )
).add_to(m_T0_final)

# T0 açık kalan kritik omurga
folium.GeoJson(
    critical_open_T0,
    name="T0 açık kalan kritik omurga yolları",
    style_function=lambda feature: {
        "color": "green",
        "weight": 4,
        "opacity": 0.90
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["road_name", "highway_clean", "mahalle_adi", "centrality_before"],
        aliases=["Kritik açık yol:", "Yol tipi:", "Mahalle:", "Normal durum centrality:"]
    )
).add_to(m_T0_final)

folium.LayerControl().add_to(m_T0_final)

try:
    add_title(
        m_T0_final,
        "#NİHAİ_HARİTA_T0 Deprem Sonrası İlk Etki",
        "Kapalı yollar, stres koridorları, yerel darboğazlar ve açık kalan kritik yol omurgası birlikte gösterilmiştir."
    )

    add_legend(
        m_T0_final,
        [
            ("blue", "Avcılar sınırı"),
            ("lightgray", "Tüm araç yol ağı"),
            ("black", "T0 kapalı yollar"),
            ("orange", "T0 ana stres koridorları"),
            ("purple", "T0 yerel darboğaz adayları"),
            ("green", "T0 açık kalan kritik omurga yolları")
        ],
        "Yeşil yollar, deprem öncesi kritik kabul edilen ve T0 anında açık kalıp özel risk sınıfına girmeyen stratejik omurga bağlantılarını gösterir."
    )
except:
    pass

m_T0_final.save("nihai_harita_T0.html")

m_T0_final

In [ ]:
# ==============================
# SİYAH VE YEŞİL YOLLAR GERÇEKTEN ÇAKIŞIYOR MU?
# ==============================

black_T0_ids = set(edges_model.loc[edges_model["closed_T0"] == True, "edge_id"])
green_T0_ids = set(critical_open_T0["edge_id"])

overlap_T0_ids = black_T0_ids.intersection(green_T0_ids)

print("T0'da hem siyah kapalı hem yeşil kritik açık görünen segment sayısı:", len(overlap_T0_ids))

if len(overlap_T0_ids) > 0:
    display(
        edges_model[
            edges_model["edge_id"].isin(overlap_T0_ids)
        ][
            [
                "edge_id",
                "road_name",
                "highway_clean",
                "mahalle_adi",
                "closed_T0",
                "centrality_before",
                "closure_probability_T0"
            ]
        ]
    )
else:
    print("Sorun yok: T0'da siyah ve yeşil aynı segment değil.")

In [ ]:
# ==============================
# HÜCRE - NİHAİ T1 HARİTASI
# ==============================

top_T1_robust_names = robust_stress_T1.head(20)["road_name"].tolist()
top_T1_bottleneck_names = local_bottlenecks_T1.head(20)["road_name"].tolist()

T1_robust_edges = stressed_edges_T1[
    stressed_edges_T1["road_name"].isin(top_T1_robust_names)
].copy()

T1_bottleneck_edges = stressed_edges_T1[
    stressed_edges_T1["road_name"].isin(top_T1_bottleneck_names)
].copy()

m_T1_final = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles="cartodbpositron"
)

# Sınır
folium.GeoJson(
    boundary,
    name="Avcılar sınırı",
    style_function=lambda feature: {
        "color": "blue",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m_T1_final)

# Tüm yol ağı
folium.GeoJson(
    edges_model,
    name="Tüm araç yol ağı",
    style_function=lambda feature: {
        "color": "lightgray",
        "weight": 1,
        "opacity": 0.20
    }
).add_to(m_T1_final)

# T1 kapalı yollar
folium.GeoJson(
    closed_T1_edges,
    name="T1 kapalı yollar",
    style_function=lambda feature: {
        "color": "black",
        "weight": 3,
        "opacity": 0.80
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["road_name", "highway_clean", "mahalle_adi", "closure_probability_T1"],
        aliases=["Kapalı yol:", "Yol tipi:", "Mahalle:", "T1 kapanma eğilimi:"]
    )
).add_to(m_T1_final)

# T1 stres koridorları
folium.GeoJson(
    T1_robust_edges,
    name="T1 ana stres koridorları",
    style_function=lambda feature: {
        "color": "orange",
        "weight": 5,
        "opacity": 0.90
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["road_name", "highway_clean", "mahalle_adi", "centrality_increase"],
        aliases=["Stres yolu:", "Yol tipi:", "Mahalle:", "Centrality artışı:"]
    )
).add_to(m_T1_final)

# T1 yerel darboğazlar
folium.GeoJson(
    T1_bottleneck_edges,
    name="T1 yerel darboğaz adayları",
    style_function=lambda feature: {
        "color": "purple",
        "weight": 4,
        "opacity": 0.90
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["road_name", "highway_clean", "mahalle_adi", "centrality_increase"],
        aliases=["Darboğaz adayı:", "Yol tipi:", "Mahalle:", "Centrality artışı:"]
    )
).add_to(m_T1_final)

# T1 açık kalan kritik omurga
folium.GeoJson(
    critical_open_T1,
    name="T1 açık kalan kritik omurga yolları",
    style_function=lambda feature: {
        "color": "green",
        "weight": 4,
        "opacity": 0.90
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["road_name", "highway_clean", "mahalle_adi", "centrality_before"],
        aliases=["Kritik açık yol:", "Yol tipi:", "Mahalle:", "Normal durum centrality:"]
    )
).add_to(m_T1_final)

folium.LayerControl().add_to(m_T1_final)

try:
    add_title(
        m_T1_final,
        "#NİHAİ_HARİTA_T1 İlk 4-5 Saat Sonrası",
        "Kapalı kalan yollar, stres koridorları, yerel darboğazlar ve açık kalan kritik yol omurgası birlikte gösterilmiştir."
    )

    add_legend(
        m_T1_final,
        [
            ("blue", "Avcılar sınırı"),
            ("lightgray", "Tüm araç yol ağı"),
            ("black", "T1 kapalı yollar"),
            ("orange", "T1 ana stres koridorları"),
            ("purple", "T1 yerel darboğaz adayları"),
            ("green", "T1 açık kalan kritik omurga yolları")
        ],
        "Yeşil yollar, deprem öncesi kritik kabul edilen ve T1 anında açık kalıp özel risk sınıfına girmeyen stratejik omurga bağlantılarını gösterir."
    )
except:
    pass

m_T1_final.save("nihai_harita_T1.html")

m_T1_final

In [ ]:
#GitHub Pages export
==============================
# COMPLETE UPDATED GITHUB PACKAGE
# Recreates index.html from scratch + maps + README.md
# Includes Summary Metrics, Sensitivity Check, and Limitations
# ==============================

import os
import shutil
import pandas as pd
import numpy as np
from google.colab import files

# ------------------------------
# 1. Output folder and filenames
# ------------------------------

web_dir = "Avcilar_Earthquake_Road_Network_Stress_Analysis_Final_GitHub"
os.makedirs(web_dir, exist_ok=True)

t0_filename = "Avcilar_T0_Immediate_Post_Earthquake_Map.html"
t1_filename = "Avcilar_T1_First_4_5_Hours_Map.html"

index_path = os.path.join(web_dir, "index.html")
readme_path = os.path.join(web_dir, "README.md")
t0_path = os.path.join(web_dir, t0_filename)
t1_path = os.path.join(web_dir, t1_filename)

# ------------------------------
# 2. Required variable checks
# ------------------------------

required_vars = [
    "m_T0_final",
    "m_T1_final",
    "edges_model",
    "robust_stress_T0",
    "robust_stress_T1",
    "local_bottlenecks_T0",
    "local_bottlenecks_T1"
]

missing_vars = [v for v in required_vars if v not in globals()]

if len(missing_vars) > 0:
    raise NameError(
        f"Missing variables: {missing_vars}. "
        "Please run the notebook cells up to the final T0/T1 map generation first."
    )

# Save maps
m_T0_final.save(t0_path)
m_T1_final.save(t1_path)

# Ensure edge_id exists
if "edge_id" not in edges_model.columns:
    edges_model = edges_model.copy()
    edges_model["edge_id"] = range(len(edges_model))

# ------------------------------
# 3. Summary metrics
# ------------------------------

def safe_bool_sum(df, col):
    if col in df.columns:
        return int(df[col].sum())
    return "N/A"

def safe_affected_neighborhoods(df, col):
    if col in df.columns and "mahalle_adi" in df.columns:
        return int(df.loc[df[col] == True, "mahalle_adi"].dropna().nunique())
    return "N/A"

total_segments = len(edges_model)
t0_closed_segments = safe_bool_sum(edges_model, "closed_T0")
t1_closed_segments = safe_bool_sum(edges_model, "closed_T1")
t1_reopened_segments = safe_bool_sum(edges_model, "reopened_T1")

t0_affected_neighborhoods = safe_affected_neighborhoods(edges_model, "closed_T0")
t1_affected_neighborhoods = safe_affected_neighborhoods(edges_model, "closed_T1")

critical_open_t0_count = len(critical_open_T0) if "critical_open_T0" in globals() else "N/A"
critical_open_t1_count = len(critical_open_T1) if "critical_open_T1" in globals() else "N/A"

metrics_df = pd.DataFrame([
    ["Total analyzed road segments", total_segments],
    ["T0 assumed closed segments", t0_closed_segments],
    ["T1 remaining closed segments", t1_closed_segments],
    ["Segments reopened by T1", t1_reopened_segments],
    ["T0 affected neighborhood count", t0_affected_neighborhoods],
    ["T1 affected neighborhood count", t1_affected_neighborhoods],
    ["T0 main stress corridor count", len(robust_stress_T0)],
    ["T1 main stress corridor count", len(robust_stress_T1)],
    ["T0 candidate local bottleneck count", len(local_bottlenecks_T0)],
    ["T1 candidate local bottleneck count", len(local_bottlenecks_T1)],
    ["T0 critical open backbone segment count", critical_open_t0_count],
    ["T1 critical open backbone segment count", critical_open_t1_count],
], columns=["Metric", "Value"])

metrics_html = metrics_df.to_html(index=False, escape=False)

# ------------------------------
# 4. Sensitivity check
# ------------------------------

def minmax_safe(series):
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    if series.max() == series.min():
        return series * 0
    return (series - series.min()) / (series.max() - series.min())

if "length_norm" in edges_model.columns:
    length_norm_for_sensitivity = edges_model["length_norm"]
else:
    length_norm_for_sensitivity = minmax_safe(edges_model["length_m"].clip(upper=500))

base_score = (
    0.50 * edges_model["building_impact_risk"] +
    0.35 * edges_model["road_type_vulnerability"] +
    0.15 * length_norm_for_sensitivity
)

top_share = 0.20
top_n = max(1, int(round(len(edges_model) * top_share)))

base_top_ids = set(
    edges_model.assign(_base_score=base_score)
    .nlargest(top_n, "_base_score")["edge_id"]
)

sensitivity_scenarios = [
    ("Base model", 0.50, 0.35, 0.15),
    ("More road-type emphasis", 0.40, 0.40, 0.20),
    ("More building-risk emphasis", 0.60, 0.25, 0.15),
    ("Balanced building and road type", 0.45, 0.45, 0.10),
]

sensitivity_rows = []

for scenario_name, w_building, w_road, w_length in sensitivity_scenarios:
    scenario_score = (
        w_building * edges_model["building_impact_risk"] +
        w_road * edges_model["road_type_vulnerability"] +
        w_length * length_norm_for_sensitivity
    )

    scenario_top_ids = set(
        edges_model.assign(_scenario_score=scenario_score)
        .nlargest(top_n, "_scenario_score")["edge_id"]
    )

    overlap_ratio = len(base_top_ids.intersection(scenario_top_ids)) / top_n

    sensitivity_rows.append({
        "Scenario": scenario_name,
        "Building risk weight": w_building,
        "Road-type vulnerability weight": w_road,
        "Length weight": w_length,
        "Top 20% overlap with base model": round(overlap_ratio, 3)
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_html = sensitivity_df.to_html(index=False, escape=False)

# ------------------------------
# 5. Create index.html from scratch
# ------------------------------

index_html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Avcılar Earthquake Road Network Stress Analysis</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background: #f7f7f7;
            color: #222;
            margin: 30px;
            line-height: 1.62;
        }}
        .container {{
            max-width: 1100px;
            margin: auto;
            background: white;
            padding: 34px;
            border-radius: 12px;
            box-shadow: 0 0 12px rgba(0,0,0,0.12);
        }}
        h1 {{
            color: #111827;
            border-bottom: 3px solid #111827;
            padding-bottom: 10px;
        }}
        h2 {{
            color: #1f2937;
            margin-top: 32px;
            border-bottom: 1px solid #e5e7eb;
            padding-bottom: 6px;
        }}
        h3 {{
            color: #374151;
            margin-top: 22px;
        }}
        .note {{
            background-color: #fff7ed;
            border-left: 5px solid #f97316;
            padding: 15px;
            margin: 20px 0;
        }}
        .method {{
            background-color: #f9fafb;
            border: 1px solid #e5e7eb;
            border-radius: 10px;
            padding: 16px;
            margin: 16px 0;
        }}
        .button-row {{
            display: flex;
            gap: 16px;
            flex-wrap: wrap;
            margin: 25px 0;
        }}
        .button {{
            display: inline-block;
            padding: 14px 20px;
            border-radius: 8px;
            text-decoration: none;
            font-weight: bold;
            color: white;
            background: #2563eb;
        }}
        .button.t1 {{
            background: #059669;
        }}
        table {{
            border-collapse: collapse;
            width: 100%;
            margin: 15px 0;
            font-size: 14px;
        }}
        th, td {{
            border: 1px solid #ddd;
            padding: 8px;
            text-align: left;
        }}
        th {{
            background-color: #f3f4f6;
        }}
        code {{
            background-color: #f3f4f6;
            padding: 2px 5px;
            border-radius: 4px;
        }}
        .signature {{
            margin-top: 36px;
            padding-top: 18px;
            border-top: 1px solid #ddd;
            font-size: 14px;
            color: #374151;
            text-align: right;
            line-height: 1.7;
        }}
    </style>
</head>
<body>
<div class="container">

<h1>Road Network Stress Analysis of Avcılar Under a Major Earthquake Scenario</h1>

<p>
This study presents a scenario-based road network stress analysis for the Avcılar district under a major earthquake scenario.
It investigates which road segments may be exposed to closure risk, which remaining open roads may experience increased network stress,
and how the road network may partially recover during the first hours after the event.
</p>

<div class="note">
<strong>Important note:</strong> This study does not provide deterministic damage or road closure predictions.
The results should be interpreted as relative risk and stress indicators produced under the adopted datasets, assumptions,
and modeling framework.
</div>

<h2>1. Interactive Maps</h2>

<p>
The interactive maps are provided as separate pages to improve loading performance. The T0 map represents the immediate
post-earthquake condition, while the T1 map represents the partial recovery stage after the first 4–5 hours of intervention.
</p>

<div class="button-row">
    <a class="button" href="{t0_filename}" target="_blank">Open T0 Map</a>
    <a class="button t1" href="{t1_filename}" target="_blank">Open T1 Map</a>
</div>

<h2>2. Methodological Overview</h2>

<div class="method">
<h3>2.1 Road network data</h3>
<p>
The vehicle road network of Avcılar was obtained from OpenStreetMap using a drive-network approach. Therefore, the analysis includes
not only major arterial roads, but also local streets, service roads, ramps, connectors, and other vehicle-accessible road segments.
The road network was converted into a graph structure, allowing the analysis of connectivity, closures, and centrality-based stress patterns.
</p>
</div>

<div class="method">
<h3>2.2 Building-related neighborhood risk</h3>
<p>
Neighborhood-level building data were used to estimate a building-related risk layer. The building stock was classified using age and floor-count indicators.
The age-related structure included pre-1980 buildings, buildings constructed between 1980 and 2000, and post-2000 buildings.
Older building ratios were treated as stronger contributors to potential earthquake-related disruption, while post-2000 buildings were used to complete
the age distribution of the building stock.
</p>
<p>
High-rise building influence was also included by separating low-rise buildings from higher floor-count categories. The resulting
<code>building_impact_risk</code> score represents a relative neighborhood-level proxy for potential building or debris-related road disruption.
</p>
</div>

<div class="method">
<h3>2.3 Road-type vulnerability</h3>
<p>
Road-type vulnerability was assigned based on the functional hierarchy of roads. Major roads such as motorway, trunk, primary, and secondary roads
were assumed to be less vulnerable and more likely to receive earlier intervention. Residential, service, and living-street segments were treated as
more vulnerable due to their local character and potentially lower intervention priority.
</p>
</div>

<div class="method">
<h3>2.4 Closure risk model</h3>
<p>
For each road segment, an earthquake-related closure risk score was calculated using building-related risk, road-type vulnerability, and normalized road length:
</p>
<p>
<code>earthquake_closure_risk = 0.50 × building_impact_risk + 0.35 × road_type_vulnerability + 0.15 × length_norm</code>
</p>
<p>
This score was then used to define the road segments assumed to be closed in the T0 scenario.
</p>
</div>

<h2>3. T0 Scenario: Immediate Post-Earthquake Condition</h2>

<p>
The T0 scenario represents the immediate impact period following the earthquake. In this map, black roads indicate road segments assumed to be closed
under the scenario. After removing these segments from the network, centrality values were recalculated for the remaining open roads.
Roads whose centrality increased after closures were interpreted as stress corridors, since they became more important for maintaining network connectivity.
</p>

<p>
In the T0 map, orange segments represent main stress corridors, purple segments represent candidate local bottlenecks, and green segments represent
critical backbone roads that remain open despite not being classified as stress or bottleneck segments.
</p>

<h2>4. T1 Scenario: First 4–5 Hours After the Earthquake</h2>

<p>
The T1 scenario represents the first 4–5 hours after the earthquake, when partial intervention and road clearance may have started.
Rather than treating T1 as an independent closure scenario, T1 was modeled as a continuation of T0. In other words, roads that remain closed in T1
were selected from the roads already closed in T0.
</p>

<p>
The T1 reopening logic considered road hierarchy, baseline network centrality, and lower building-related risk. Major and more central roads were assumed
to have higher reopening priority, while roads with higher closure risk and lower reopening priority were more likely to remain closed.
</p>

<h2>5. Interpretation of Map Colors</h2>

<table>
<tr><th>Color</th><th>Meaning</th><th>Interpretation</th></tr>
<tr><td><strong>Black</strong></td><td>Roads assumed to be closed</td><td>Segments removed from the network under the scenario.</td></tr>
<tr><td><strong>Orange</strong></td><td>Main stress corridors</td><td>Open roads whose centrality increased after closures.</td></tr>
<tr><td><strong>Purple</strong></td><td>Candidate local bottlenecks</td><td>Short but potentially critical local links with increased importance.</td></tr>
<tr><td><strong>Green</strong></td><td>Critical open backbone roads</td><td>Initially important roads that remain open and are not classified as stress or bottleneck segments.</td></tr>
<tr><td><strong>Gray</strong></td><td>Background road network</td><td>Shown through the base map. Not all gray roads are analytical outputs.</td></tr>
</table>

<h2>6. Use of the Outputs</h2>

<p>
The outputs can be used as a decision-support layer for post-earthquake transportation planning. The maps help identify potential closure-prone segments,
open roads likely to experience increased load, local bottleneck candidates, and critical backbone roads that remain available under the scenario.
</p>

<p>
The analysis is most useful for asking relative and operational questions such as:
</p>

<ul>
    <li>Which roads stand out in terms of closure risk or structural vulnerability?</li>
    <li>Which open roads may carry additional network stress after disruptions?</li>
    <li>Which local links may become bottlenecks?</li>
    <li>Which open roads continue to serve as critical network backbone segments?</li>
    <li>How does the network change from T0 to T1?</li>
</ul>

<h2>7. Summary Metrics</h2>

<p>
The table below summarizes the main quantitative outputs generated by the scenario-based analysis.
</p>

{metrics_html}

<h2>8. Sensitivity Check</h2>

<p>
A simple sensitivity check was conducted to test whether the high-risk road segment ranking is strongly dependent on the selected closure-risk weights.
The base model uses building-related risk, road-type vulnerability, and normalized road length with weights of 0.50, 0.35, and 0.15, respectively.
Alternative weighting schemes were compared against the base model by measuring the overlap of the top 20% highest-risk road segments.
</p>

{sensitivity_html}

<p>
A higher overlap value indicates that the high-risk segment set remains relatively stable under alternative assumptions.
This check does not replace a full uncertainty analysis, but it provides an initial robustness assessment for the closure-risk logic.
</p>

<h2>9. Limitations</h2>

<p>
This analysis should be interpreted as a scenario-based decision-support study rather than a deterministic damage or traffic forecast.
Several important limitations should be considered when interpreting the outputs:
</p>

<ul>
    <li>The building-related risk layer is based on neighborhood-level building indicators, not building-by-building structural damage predictions.</li>
    <li>The model does not include real-time traffic volume, post-earthquake congestion, or individual route choice behavior.</li>
    <li>Road width, lane count, bridge and viaduct vulnerability, underpass conditions, and detailed engineering fragility were not directly modeled.</li>
    <li>Emergency response vehicle locations, road-clearing team availability, and actual operational constraints were not included as explicit inputs.</li>
    <li>The T1 recovery logic is based on assumed reopening priority using road hierarchy, baseline centrality, and lower building-related risk; it is not an observed recovery process.</li>
    <li>The maps should therefore be interpreted as relative stress and vulnerability indicators under the selected assumptions, not as definitive lists of closed or safe roads.</li>
</ul>

<div class="note">
The maps should not be interpreted as definitive lists of safe or closed roads. They are scenario-based indicators derived from the selected data,
filters, and assumptions.
</div>

<div class="signature">
    <div style="font-size: 13px; color: #6b7280; margin-bottom: 4px;">Prepared by</div>
    <strong>Muhammet Yusuf Demir</strong><br>
    Istanbul Technical University<br>
    Industrial Engineering
</div>

</div>
</body>
</html>
"""

with open(index_path, "w", encoding="utf-8") as f:
    f.write(index_html)

# ------------------------------
# 6. Create README.md
# ------------------------------

readme_content = """# Avcılar Earthquake Road Network Stress Analysis

This repository contains an interactive web-based report for a scenario-based road network stress analysis of Avcılar under a major earthquake scenario.

## Project Scope

The study investigates:

- road segments that may be exposed to closure risk under a major earthquake scenario,
- open road segments that may experience increased network stress after disruptions,
- candidate local bottlenecks,
- critical open backbone roads,
- and the difference between the immediate post-earthquake condition (T0) and the partial recovery stage after the first 4–5 hours (T1).

## Methodological Summary

The analysis uses:

- OpenStreetMap-based vehicle road network data,
- neighborhood-level building indicators,
- building age and floor-count based risk proxies,
- road-type vulnerability assumptions,
- normalized road length,
- graph-based betweenness centrality,
- T0 closure scenario logic,
- T1 reopening and remaining-closure logic.

The closure-risk model combines building-related neighborhood risk, road-type vulnerability, and normalized road length. After assumed closures are removed from the network, centrality is recalculated to identify stress corridors and candidate bottlenecks.

## Scenario Definitions

- **T0:** Immediate post-earthquake condition.
- **T1:** First 4–5 hours after the earthquake, representing a partial recovery stage in which some roads may have been reopened depending on road hierarchy, network centrality, and lower building-related risk.

## Map Layers

- **Black:** roads assumed to be closed,
- **Orange:** main stress corridors,
- **Purple:** candidate local bottlenecks,
- **Green:** critical open backbone roads,
- **Gray:** background road network.

## Limitations

This analysis is not a deterministic damage or road closure prediction. It does not include real-time traffic volumes, observed earthquake damage, detailed structural vulnerability of bridges or viaducts, emergency response team locations, or actual operational road-clearing constraints. The outputs should be interpreted as relative risk and stress indicators under the adopted assumptions.

## Prepared by

**Muhammet Yusuf Demir**
Istanbul Technical University
Industrial Engineering
"""

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

# ------------------------------
# 7. Create zip
# ------------------------------

zip_path = shutil.make_archive(web_dir, "zip", web_dir)

print("Final updated package created successfully.")
print("Folder:", web_dir)
print("Files to upload/update on GitHub:")
print("index.html")
print("README.md")
print(t0_filename)
print(t1_filename)
print("ZIP:", zip_path)

files.download(zip_path)